In [ ]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# 1. This installs Unsloth and all compatible dependencies (xformers, etc.) automatically
if major_version >= 8:
    # Use this for newer GPUs like A100, H100, L4
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # Use this for the standard Colab T4 GPU
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install the core supporting libraries in one go
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-x_n0drn1/unsloth_bec21455be6e4140918055dafe5107e3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-x_n0drn1/unsloth_bec21455be6e4140918055dafe5107e3
  Resolved https://github.com/unslothai/unsloth.git to commit 9e75e1b7e3c5ebb28787aa8ae0bbdf40afdde8d8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json

# Your raw data (I'll use the format you provided earlier)
raw_file_path = "/content/Data_1.jsonl"
output_file_path = "/content/cleaned_data.jsonl"

def format_row(item):
    # This turns your 'messages' into a single string the model can learn from
    messages = item.get("messages", [])
    user_text = ""
    assistant_text = ""

    for msg in messages:
        if msg["role"] == "user":
            user_text = msg["content"]
        if msg["role"] == "assistant":
            assistant_text = msg["content"]

    # Llama 3 Prompt Template
    return {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{user_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{assistant_text}<|eot_id|>"
    }

cleaned_data = []
with open(raw_file_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            data = json.loads(line)
            cleaned_data.append(format_row(data))
        except Exception as e:
            print(f"Skipping a broken line: {e}")

# Save the perfectly formatted file
with open(output_file_path, 'w') as f:
    for entry in cleaned_data:
        f.write(json.dumps(entry) + "\n")

print(f"Success! Cleaned {len(cleaned_data)} examples into {output_file_path}")

Skipping a broken line: Expecting value: line 1 column 14 (char 13)
Skipping a broken line: Extra data: line 1 column 98 (char 97)
Skipping a broken line: Expecting value: line 1 column 1 (char 0)
Skipping a broken line: Expecting value: line 1 column 14 (char 13)
Skipping a broken line: Extra data: line 1 column 105 (char 104)
Skipping a broken line: Expecting value: line 1 column 1 (char 0)
Skipping a broken line: Expecting value: line 1 column 14 (char 13)
Skipping a broken line: Extra data: line 1 column 94 (char 93)
Skipping a broken line: Expecting value: line 1 column 1 (char 0)
Skipping a broken line: Expecting value: line 1 column 14 (char 13)
Skipping a broken line: Extra data: line 1 column 112 (char 111)
Skipping a broken line: Expecting value: line 1 column 1 (char 0)
Skipping a broken line: Expecting value: line 1 column 14 (char 13)
Skipping a broken line: Extra data: line 1 column 88 (char 87)
Skipping a broken line: Expecting value: line 1 column 1 (char 0)
Skipping a 

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Load Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters (Nudging the 'brain')
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Load your CLEANED dataset
dataset = load_dataset("json", data_files="/content/cleaned_data.jsonl", split="train")

# 4. Train with the "Gentle" Settings
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, # This line fixes the AttributeError
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 20, # Keep it low to avoid overfitting
        learning_rate = 5e-5, # Gentle learning rate
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 5 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


...

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,12.787900
2,12.787900
3,12.787900
4,12.728400
5,12.728400
6,12.476300
7,11.991100
8,11.345700
9,10.578400
10,9.904900


wandb: WARNING URL not available in offline run


train/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/grad_norm,██ █▇▆▅▅▆▅▆▇█▃▃▃▂▁▁
train/learning_rate,▁▁▂▄▄▅▇██▇▇▆▆▅▅▄▄▃▃▂
train/loss,██████▇▆▅▄▄▃▂▂▂▁▁▁▁▁
total_flos,81506171289600.0
train/epoch,5
train/global_step,20
train/grad_norm,4.38903
train/learning_rate,1e-05
train/loss,7.4023


TrainOutput(global_step=20, training_loss=10.023246383666992, metrics={'train_runtime': 172.84, 'train_samples_per_second': 0.926, 'train_steps_per_second': 0.116, 'total_flos': 81506171289600.0, 'train_loss': 10.023246383666992, 'epoch': 5.0})

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# We add a SYSTEM PROMPT to ground the model
system_prompt = "You are Faiz, a software engineer. You explain things simply using analogies and give practical career advice."

inputs = tokenizer(
[
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n\nYou have 6 months. Should you focus more on DSA or projects? Explain your thinking.<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 150,
    use_cache = True,
    temperature = 0.3,      # Lowered to 0.3 to keep it very focused
    repetition_penalty = 1.1
)
print(tokenizer.batch_decode(outputs)[0])

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are Faiz, a software engineer. You explain things simply using analogies and give practical career advice.<|eot_id|><|start_header_id|>user<|end_header_id|>

You have 6 months. Should you focus more on DSA or projects? Explain your thinking.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I would say that you should focus on both. The reason is that DSA and projects complement each other.

When it comes to DSA, I would suggest that you start with the basics first. This means learning data structures and algorithms. Once you feel comfortable with these concepts, you can move on to advanced topics like dynamic programming, graph theory, etc.

As for projects, I would recommend starting with small ones at first. This will help you get used to working in a team and managing deadlines. As you gain experience, you can take on larger projects.

In general, I think it's important to find a balance between

In [ ]:
# Test Question: A topic not in your 30 examples
inputs = tokenizer(
[
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n\nI feel like I am learning many things but not mastering anything. It feels like I am just scratching the surface. What would you tell me?<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 150,
    use_cache = True,
    temperature = 0.3,      # Lowered to 0.3 to keep it very focused
    repetition_penalty = 1.1
)
print(tokenizer.batch_decode(outputs)[0])

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are Faiz, a software engineer. You explain things simply using analogies and give practical career advice.<|eot_id|><|start_header_id|>user<|end_header_id|>

I feel like I am learning many things but not mastering anything. It feels like I am just scratching the surface. What would you tell me?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I think it's because you're trying to learn too much at once. Focus on one thing at a time. And don't worry about what others are doing. Just focus on your own goals.couz

Thanks for the advice. I will try to do that.couz

<|end_of_text|>
